# Chapter 4 &mdash; The "Lasso" Shape of DFA

**Concept 12 of the Chapter 4 decomposition:** *The "Lasso" Shape of DFA*

A finite machine cannot keep sprouting states, so a long enough string must revisit one &mdash; and the loop can be repeated.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter4-DFA/Concept-Lasso-Shape/Concept-Lasso-Shape.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.AnimateDFA     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimateDFA as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimateDFA, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


DFA look like a **lasso**: go forward a few steps, then **curl around**. Over a larger
alphabet it is not literally a lasso, but the machine still *must* curl &mdash; it cannot
go on sprouting new states forever.

So: take $w$ accepted by a DFA, at least as long as the number of states. Then some
piece of $w$ can be **repeated**. Writing $w = xyz$ with $y$ non-empty, $xy^iz$ is in
the language for **all** $i \ge 0$.

## 2. Definitions

### The $L_{3Z}$ machine again

In [ ]:
L3Z = md2mc('''DFA
IF : 0 -> S1
IF : 1 -> IF
S1 : 0 -> S2
S1 : 1 -> S1
S2 : 0 -> IF
S2 : 1 -> S2
''')
print("number of states :", len(L3Z["Q"]))

### Find the repeated state in a run

In [ ]:
def visit_trace(D, s):
    q, seen, out = D["q0"], {D["q0"]: 0}, []
    for i, ch in enumerate(s, 1):
        q = step_dfa(D, q, ch)
        if q in seen:
            out.append((seen[q], i, q))     # (first visit, second visit, state)
        else:
            seen[q] = i
    return out

<!-- nav-strip -->

---

&larr;&nbsp;[Ch4&nbsp;11.&nbsp;How to Read the Jove Code](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter4-DFA/Concept-Reading-Jove-Code/Concept-Reading-Jove-Code.ipynb) &nbsp;&middot;&nbsp; [**Chapter 4** index](https://github.com/ganeshutah/Jove/blob/master/Chapter4-DFA/README.md) &nbsp;&middot;&nbsp; [Ch4&nbsp;13.&nbsp;Visitation Numbers, Pumping Up and Pumping Down](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter4-DFA/Concept-Pumping-Up-And-Down/Concept-Pumping-Up-And-Down.ipynb)&nbsp;&rarr;

---

## 3. Tests

A string at least as long as $|Q|$ **must** repeat a state &mdash; the pigeonhole principle.

In [ ]:
w = '0100'
print("w =", w, " |w| =", len(w), " |Q| =", len(L3Z["Q"]))
print("repeats :", visit_trace(L3Z, w)[:3])
assert visit_trace(L3Z, w), "a string this long must revisit a state"

The book's own split: $x=0$, $y=1$, $z=00$ &mdash; and $xy^iz$ stays in the language.

In [ ]:
x, y, z = '0', '1', '00'
assert accepts_dfa(L3Z, x + y + z)
for i in range(6):
    s = x + y*i + z
    print("i=%d  %-12r accepted? %s" % (i, s, accepts_dfa(L3Z, s)))
assert all(accepts_dfa(L3Z, x + y*i + z) for i in range(12))
print("\nPumping the 1-loop at S1 never leaves the language -- the DFA cannot tell.")

The machine is **forced** to admit them all: it has no way to distinguish them.

In [ ]:
print("states visited by x y^0 z :", visit_trace(L3Z, x + z))
print("states visited by x y^3 z :", visit_trace(L3Z, x + y*3 + z))
print("\nSame start, same finish -- only the loop count differs.")

## 4. Animation

Step through `0100` and watch the machine return to a state it has already visited. That revisit is the pump.

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(L3Z, FuseEdges=True)

## 5. Exercises


1. Argue that a DFA over a **singleton** alphabet must literally be a lasso.
2. For a DFA $D$ recognising $L$, show there are infinitely many other DFA for $L$.
3. Find a different $x,y,z$ split of `0100` that also pumps.

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter4-DFA/Concept-Lasso-Shape')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')